# Comparison of biofuel SAF impacts

The environmental impacts of SAFs can vary significantly depending on the feedstock, production pathway and allocation method (i.e., how impacts are distributed among co-products at the different stages of the life cycle). Here, we compare various LCA models available for biofuels from the *premise* library:

Dataset | Reference
--- | ---
Kerosene production, from methanol, from wood, with CCS, economic allocation | Hank et al. (2019)
Kerosene production, synthetic, Fischer Tropsch process, hydrogen from wood gasification, energy allocation | van Der Giesen et al. (2014)
Kerosene production, via Fischer-Tropsch, from forest residues, energy allocation | Cavalett & Cherubini (2022)
Kerosene production, via Fischer-Tropsch, from forest product (non-residual), energy allocation | Cavalett & Cherubini (2022)

In [ ]:
# --- Import libraries ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import lca_algebraic as agb
import brightway2 as bw

# --- Setup brightway project (see LCA yaml configuration file) ---
bw.projects.set_current("ecats2026")

# --- Select three different LCA models for describing biofuels (production only) ---
biofuel_methanol_wood = agb.findActivity(
    "kerosene production, from methanol, from wood, with CCS, economic allocation",
    loc="RER",
    db_name="ecoinvent_cutoff_3.12_remind_SSP2-PkBudg1000_2020",
)

biofuel_fischer_tropsch_wood_gas = agb.findActivity(
    "kerosene production, synthetic, Fischer Tropsch process, hydrogen from wood gasification, energy allocation",
    loc="RER",
    db_name="ecoinvent_cutoff_3.12_remind_SSP2-PkBudg1000_2020",
)

biofuel_fischer_tropsch_residues = agb.findActivity(
    "kerosene production, via Fischer-Tropsch, from forest residues, energy allocation",
    loc="RER",
    db_name="ecoinvent_cutoff_3.12_remind_SSP2-PkBudg1000_2020",
)

biofuel_fischer_tropsch_products = agb.findActivity(
    "kerosene production, via Fischer-Tropsch, from forest product (non-residual), energy allocation",
    loc="RER",
    db_name="ecoinvent_cutoff_3.12_remind_SSP2-PkBudg1000_2020",
)

# --- Impacts to evaluate ---
impacts = [
    ("ReCiPe 2016 v1.03, midpoint (H)", "land use", "agricultural land occupation (LOP)"),
    ("ReCiPe Aviation", "climate change", "global warming potential (GWP100)"),
    ("ReCiPe 2016 v1.03, midpoint (H)", "water use", "water consumption potential (WCP)"),
    ("ReCiPe Aviation", "total: ecosystem quality", "ecosystem quality (GWP100)"),
    ("ReCiPe Aviation", "total: human health", "human health (GWP100)"),
]

In [ ]:
# --- Compute impacts ---
df = agb.compute_impacts(
    [
        biofuel_methanol_wood,
        biofuel_fischer_tropsch_wood_gas,
        biofuel_fischer_tropsch_residues,
        biofuel_fischer_tropsch_products,
    ],
    impacts,
)

# --- Normalise to set on common scale ---
df_norm = df / df.max()

# --- Radar/Spider plot ---
labels = [
    "Land Use",
    "Climate Change\n(GWP100)",
    "Water Use",
    "Ecosystem\nDamages",
    "Human Health\nDamages",
]  # df_norm.columns
num_vars = len(labels)
angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
angles += angles[:1]  # close the circle
fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
for idx, row in df_norm.iterrows():
    values = row.tolist()
    values += values[:1]
    ax.plot(angles, values, linewidth=2, label=idx)
    ax.fill(angles, values, alpha=0.1)
ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)
ax.set_thetagrids(np.degrees(angles[:-1]), labels)
for label, angle in zip(ax.get_xticklabels(), angles):
    if angle == 0 or angle == np.pi:
        label.set_horizontalalignment("center")
    elif 0 < angle < np.pi:
        label.set_horizontalalignment("left")
    else:
        label.set_horizontalalignment("right")
    label.set_verticalalignment("bottom")
# ax.set_rlabel_position(0)
ax.set_ylim(0, 1)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.0), ncol=1, frameon=False)
plt.show()
plt.savefig("biofuels_lca_comparison.pdf")